In [ ]:
# ==========================================
# GUESTHOUSE AI SCHEDULER - LIVE DEMO (v9)
# ==========================================

import gspread
import pandas as pd
import time
from google.colab import auth
from google.auth import default

# --- 1. AUTORYZACJA I POŁĄCZENIE Z BAZĄ ---
print("🔄 Autoryzacja z Google Sheets...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

NAZWA_PLIKU = "AI_Guesthouse_Scheduler_v1"
sh = gc.open(NAZWA_PLIKU)
print("✅ Połączono z bazą danych!")

# --- 2. GŁÓWNY SILNIK (SPRAWDZANIE LIMITÓW, KONFLIKTÓW I GRAFIKU) ---
def super_system_ai_v9():
    try:
        # Pobieranie danych
        ws_requests = sh.worksheet("Wnioski_Urlopowe")
        ws_employees = sh.worksheet("Pracownicy")
        ws_grafik = sh.worksheet("Grafik_Sierpien_2026")

        requests_df = pd.DataFrame(ws_requests.get_all_records())
        if requests_df.empty:
            return False # Brak danych do analizy

        # Szukamy, czy są jakiekolwiek NOWE wnioski do przetworzenia
        nowe_wnioski = requests_df[requests_df['Status'].astype(str).str.strip().isin(['', 'Oczekuje'])]
        if nowe_wnioski.empty:
            return False # Brak nowych wniosków

        print("\n🤖 AI: Rozpoczynam weryfikację nowych wniosków...")

        employees_df = pd.DataFrame(ws_employees.get_all_records())
        grafik_data = ws_grafik.get_all_values()

        # Dopasowanie kolumn
        employees_df.columns = [c.strip() for c in employees_df.columns]
        col_ytd = [c for c in employees_df.columns if "Wykorzy" in c and "YTD" in c][0]
        col_limit = [c for c in employees_df.columns if "Limit" in c][0]

        urlopy_dzienne = {}
        header_grafik = grafik_data[0]
        grafik_rows = grafik_data[1:]
        zmiany_w_bazie = False

        # Zliczamy już zaakceptowane urlopy, żeby nowy wniosek nie nałożył się na stare!
        for i, row in requests_df.iterrows():
            if str(row.get('Status', '')).strip() == 'ZAAKCEPTOWANY':
                dzial = str(row['ID']).strip()[0]
                for d in range(int(row['OD']), int(row['DO']) + 1):
                    klucz = f"{dzial}_{d}"
                    urlopy_dzienne[klucz] = urlopy_dzienne.get(klucz, 0) + 1

        # Przetwarzanie wniosków
        for i, row in requests_df.iterrows():
            status_aktualny = str(row.get('Status', '')).strip()

            # Przetwarzamy TYLKO puste lub oczekujące
            if status_aktualny not in ['', 'Oczekuje']:
                continue

            if str(row.get('OD', '')).strip() == "" or str(row.get('DO', '')).strip() == "":
                continue

            pracownik_id = str(row['ID']).strip()

            try:
                od_dnia = int(row['OD'])
                do_dnia = int(row['DO'])
            except ValueError:
                requests_df.at[i, 'Status'] = "BŁĄD DANYCH: Wpisz liczby!"
                zmiany_w_bazie = True
                continue

            dni_wniosku = do_dnia - od_dnia + 1
            dzial = pracownik_id[0]

            # --- A. SPRAWDZANIE LIMITU URLOPOWEGO ---
            emp_info = employees_df[employees_df['ID'].astype(str).str.strip() == pracownik_id]
            if emp_info.empty:
                requests_df.at[i, 'Status'] = "BŁĄD: Brak pracownika w bazie"
                zmiany_w_bazie = True
                continue

            limit = int(emp_info.iloc[0][col_limit])
            wykorzystano = int(emp_info.iloc[0][col_ytd])
            pozostalo = limit - wykorzystano

            if dni_wniosku > pozostalo:
                requests_df.at[i, 'Status'] = f"ODRZUCONY (Brak dni, zostało: {pozostalo})"
                print(f"❌ {pracownik_id}: Brak dni. Chce {dni_wniosku}, ma {pozostalo}")
                zmiany_w_bazie = True
                continue

            # --- B. SPRAWDZANIE OBSADY (Wykrywanie konfliktów w dziale) ---
            konflikt = False
            dzien_konfliktu = None
            for d in range(od_dnia, do_dnia + 1):
                klucz_dnia = f"{dzial}_{d}"
                # Maksymalnie 2 osoby z tego samego działu na urlopie
                if urlopy_dzienne.get(klucz_dnia, 0) >= 2:
                    konflikt = True
                    dzien_konfliktu = d
                    break

            if konflikt:
                requests_df.at[i, 'Status'] = f'ODRZUCONY (Konflikt w dniu: {dzien_konfliktu}.08)'
                print(f"❌ {pracownik_id}: Odrzucono - brak obsady w dniu {dzien_konfliktu}")
                zmiany_w_bazie = True
                continue

            # --- C. AKCEPTACJA I NANIESIENIE NA GRAFIK ---
            requests_df.at[i, 'Status'] = "ZAAKCEPTOWANY"
            print(f"✅ {pracownik_id}: Zaakceptowano urlop! (Zostanie dni: {pozostalo - dni_wniosku})")
            zmiany_w_bazie = True

            # Rezerwujemy miejsce dla tego urlopu
            for d in range(od_dnia, do_dnia + 1):
                klucz_dnia = f"{dzial}_{d}"
                urlopy_dzienne[klucz_dnia] = urlopy_dzienne.get(klucz_dnia, 0) + 1

            # Wpisujemy 'U' do grafiku w pamięci
            for g_row in grafik_rows:
                if str(g_row[0]).strip() == pracownik_id:
                    # Dodajemy puste komórki, jeśli wiersz jest za krótki
                    while len(g_row) <= 31:
                        g_row.append('')

                    for d in range(od_dnia, do_dnia + 1):
                        if 1 <= d <= 31:
                            g_row[d] = 'U'

        # ZAPIS DO ARKUSZY (Jeśli były jakiekolwiek zmiany)
        if zmiany_w_bazie:
            ws_requests.update([requests_df.columns.values.tolist()] + requests_df.values.tolist())
            ws_grafik.update([header_grafik] + grafik_rows)
            print("🎉 BAZA I GRAFIK ZSYNCHRONIZOWANE!\n")
            return True

        return False

    except Exception as e:
        print(f"❌ Wystąpił błąd podczas analizy: {e}")
        return False

# --- 3. GŁÓWNA PĘTLA LIVE (NASŁUCHIWANIE APPSHEET) ---
print("\n🚀 SYSTEM AI W TRYBIE LIVE (Nasłuchuję zmian w AppSheet...)")
print("Naciśnij 'Stop' w Colabie, aby zakończyć.\n")

try:
    while True:
        # Pętla po prostu odpala główny silnik i czeka
        byla_aktualizacja = super_system_ai_v9()

        if byla_aktualizacja:
            print("😴 Czekam na kolejne wnioski...\n")

        # 5 sekund przerwy, żeby nie naruszyć limitów Google
        time.sleep(5)

except KeyboardInterrupt:
    print("\n🛑 Tryb LIVE wyłączony. Dziękuję za pracę.")
except Exception as e:
    print(f"\n⚠️ Wystąpił niespodziewany błąd pętli: {e}")

🔄 Autoryzacja z Google Sheets...
✅ Połączono z bazą danych!

🚀 SYSTEM AI W TRYBIE LIVE (Nasłuchuję zmian w AppSheet...)
Naciśnij 'Stop' w Colabie, aby zakończyć.

